In [14]:
import os
import re
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torch_xla
import torch_xla.core.xla_model as xm

from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

In [15]:
CONFIG = {
    'model_name'  : 'xlm-roberta-base',
    'max_len'     : 128,
    'batch_size'  : 32,
    'epochs'      : 4,
    'lr'          : 2e-5,
    'warmup_ratio': 0.1,
    'test_size'   : 0.20,
    'random_seed' : 42,
    'save_path'   : 'xlmroberta_urdu_best.pt',
}

torch.manual_seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])

# Get TPU device once globally
DEVICE = xm.xla_device()
print(f"TPU Device : {DEVICE}")
print(f"XLA Device type: {xm.xla_device_hw(DEVICE)}")

TPU Device : xla:0
XLA Device type: TPU


In [16]:
print("\n" + "="*55)
print("STEP 1: Loading and cleaning data...")
print("="*55)

df = pd.read_csv('/content/combined_urdu_news.csv')

label_map = {
    'TRUE': 0, 'TRUE ': 0, 'REAL': 0, 'real': 0, '1': 0,
    'FAKE': 1, 'FAKE ': 1, 'fake': 1, 'fake ': 1,
}
df['Label'] = df['Label'].str.strip()
df['label'] = df['Label'].map(label_map)
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)
df = df.rename(columns={'News Items': 'text'})[['text', 'label']]
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
df = df[df['text'].str.split().str.len() >= 5].reset_index(drop=True)

def urdu_ratio(text):
    urdu = len(re.findall(r'[\u0600-\u06FF]', str(text)))
    return urdu / (len(str(text)) + 1)

df = df[df['text'].apply(urdu_ratio) >= 0.5].reset_index(drop=True)

print(f"Total rows : {len(df)}")
print(f"Real  (0)  : {(df['label']==0).sum()}")
print(f"Fake  (1)  : {(df['label']==1).sum()}")
print(f"Imbalance  : {(df['label']==0).sum()/(df['label']==1).sum():.2f}x")


STEP 1: Loading and cleaning data...
Total rows : 62017
Real  (0)  : 42328
Fake  (1)  : 19689
Imbalance  : 2.15x


In [17]:
def preprocess_urdu(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F\u0670]', '', text)
    text = text.replace('\u0649', '\u06CC')
    text = text.replace('\u0643', '\u06A9')
    text = text.replace('\u0647', '\u06BE')
    text = re.sub(r'http\S+|www\S+|@\S+|#\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(preprocess_urdu)
df = df[df['text'].str.strip().str.len() > 0].reset_index(drop=True)

In [18]:
print("\nSTEP 3: Train/test split...")

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size    = CONFIG['test_size'],
    random_state = CONFIG['random_seed'],
    stratify     = df['label'].values,
)

X_train = df['text'].values[train_idx]
y_train = df['label'].values[train_idx]
X_test  = df['text'].values[test_idx]
y_test  = df['label'].values[test_idx]

print(f"Train : {len(X_train)}  |  Test : {len(X_test)}")


STEP 3: Train/test split...
Train : 49613  |  Test : 12404


In [19]:
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)

# Move class weights directly to TPU device
class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
print(f"\nClass weights -> Real: {cw[0]:.4f}  Fake: {cw[1]:.4f}")


Class weights -> Real: 0.7326  Fake: 1.5749


In [20]:
print("\nSTEP 5: Tokenizing dataset...")

tokenizer = XLMRobertaTokenizer.from_pretrained(CONFIG['model_name'])

class UrduNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        # Pre-tokenize everything upfront — faster than tokenizing per batch
        print(f"  Pre-tokenizing {len(texts)} samples...", flush=True)
        self.encodings = tokenizer(
            list(texts),
            max_length     = max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'     : self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label'         : self.labels[idx],
        }

train_dataset = UrduNewsDataset(X_train, y_train, tokenizer, CONFIG['max_len'])
test_dataset  = UrduNewsDataset(X_test,  y_test,  tokenizer, CONFIG['max_len'])

# Plain DataLoader — tensors are moved to TPU inside the training loop
train_loader = DataLoader(
    train_dataset,
    batch_size  = CONFIG['batch_size'],
    shuffle     = True,
    drop_last   = True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size  = CONFIG['batch_size'],
    shuffle     = False,
    drop_last   = False,
)

print(f"  Train batches : {len(train_loader)}")
print(f"  Test  batches : {len(test_loader)}")



STEP 5: Tokenizing dataset...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

  Pre-tokenizing 49613 samples...
  Pre-tokenizing 12404 samples...
  Train batches : 1550
  Test  batches : 388


In [21]:
print("\nSTEP 6: Loading model onto TPU...")

model = XLMRobertaForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels = 2,
)

# Move entire model to TPU device
model = model.to(DEVICE)

# Verify model is on TPU
sample_param = next(model.parameters())
print(f"  Model device : {sample_param.device}")
print(f"  Parameters   : {sum(p.numel() for p in model.parameters()):,}")


STEP 6: Loading model onto TPU...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Model device : xla:0
  Parameters   : 278,045,186


In [22]:
optimizer    = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
total_steps  = len(train_loader) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [24]:
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    all_preds  = []
    all_labels = []

    for step, batch in enumerate(loader):
        # Move each tensor to TPU explicitly
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # THIS IS THE KEY LINE — tells XLA to execute the graph on TPU now
        # Without this, XLA keeps building the graph and never runs it
        xm.mark_step()

        scheduler.step()

        # Move results back to CPU for metric computation
        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.detach().cpu().numpy())

        if (step + 1) % 100 == 0:
            print(f"    Step {step+1}/{len(loader)} | Loss: {total_loss/(step+1):.4f}", flush=True)

    return (
        total_loss / len(loader),
        f1_score(all_labels, all_preds, average='macro'),
        accuracy_score(all_labels, all_preds),
    )

In [25]:
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)

            # Must call after every forward pass on TPU
            xm.mark_step()

            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.detach().cpu().numpy())

    return (
        total_loss / len(loader),
        f1_score(all_labels, all_preds, average='macro'),
        accuracy_score(all_labels, all_preds),
        all_preds,
        all_labels,
    )

In [26]:
print("\nSTEP 9: Training...")
print(f"{'Epoch':<8} {'Train Loss':>11} {'Train F1':>10} {'Val Loss':>10} {'Val F1':>8} {'Val Acc':>8}")
print("-" * 58)

best_f1 = 0.0
history = []

for epoch in range(1, CONFIG['epochs'] + 1):
    print(f"\nEpoch {epoch}/{CONFIG['epochs']}")

    train_loss, train_f1, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, criterion
    )
    val_loss, val_f1, val_acc, val_preds, val_labels = eval_epoch(
        model, test_loader, criterion
    )

    history.append({
        'epoch': epoch, 'train_loss': train_loss,
        'train_f1': train_f1, 'val_loss': val_loss,
        'val_f1': val_f1, 'val_acc': val_acc,
    })

    print(f"{epoch:<8} {train_loss:>11.4f} {train_f1:>10.4f} "
          f"{val_loss:>10.4f} {val_f1:>8.4f} {val_acc:>8.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        # xm.save is the TPU-safe version of torch.save
        xm.save(model.state_dict(), CONFIG['save_path'])
        print(f"  -> Saved best model (val F1 = {best_f1:.4f})")


STEP 9: Training...
Epoch     Train Loss   Train F1   Val Loss   Val F1  Val Acc
----------------------------------------------------------

Epoch 1/4
    Step 100/1550 | Loss: 0.7036
    Step 200/1550 | Loss: 0.6345
    Step 300/1550 | Loss: 0.5510
    Step 400/1550 | Loss: 0.4940
    Step 500/1550 | Loss: 0.4523
    Step 600/1550 | Loss: 0.4192
    Step 700/1550 | Loss: 0.3945
    Step 800/1550 | Loss: 0.3743
    Step 900/1550 | Loss: 0.3595
    Step 1000/1550 | Loss: 0.3445
    Step 1100/1550 | Loss: 0.3324
    Step 1200/1550 | Loss: 0.3223
    Step 1300/1550 | Loss: 0.3112
    Step 1400/1550 | Loss: 0.3014
    Step 1500/1550 | Loss: 0.2919
1             0.2872     0.8773     0.1861   0.9447   0.9530
  -> Saved best model (val F1 = 0.9447)

Epoch 2/4
    Step 100/1550 | Loss: 0.1622
    Step 200/1550 | Loss: 0.1605
    Step 300/1550 | Loss: 0.1692
    Step 400/1550 | Loss: 0.1611
    Step 500/1550 | Loss: 0.1576
    Step 600/1550 | Loss: 0.1550
    Step 700/1550 | Loss: 0.1564
    

In [32]:
def predict_news(raw_text):
    model.eval()
    cleaned  = preprocess_urdu(raw_text)
    encoding = tokenizer(
        cleaned,
        max_length     = CONFIG['max_len'],
        padding        = 'max_length',
        truncation     = True,
        return_tensors = 'pt',
    )
    input_ids      = encoding['input_ids'].to(DEVICE)
    attention_mask = encoding['attention_mask'].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)
        xm.mark_step()
        probs   = probs.cpu().numpy()[0]
        pred    = int(np.argmax(probs))

    return {
        'prediction': 'FAKE' if pred == 1 else 'REAL',
        'confidence': round(float(probs[pred]), 4),
        'prob_real' : round(float(probs[0]), 4),
        'prob_fake' : round(float(probs[1]), 4),
    }

print("\nDone. To predict:")
result = predict_news('پاکستان کرکٹ بورڈ نے قومی ٹیم کے نئے کوچ کا اعلان کر دیا۔ نئے کوچ اگلے ماہ سے اپنی ذمہ داریاں سنبھالیں گے۔')
print(result)


result = predict_news('ذرائع کے مطابق کہا جا رہا ہے کہ پاکستان میں کل سے تمام اسکول ہمیشہ کے لیے بند کر دیے جائیں گے۔ یہ خبر سوشل میڈیا پر تیزی سے وائرل ہو رہی ہے۔')
print(result)


Done. To predict:
{'prediction': 'REAL', 'confidence': 0.9973, 'prob_real': 0.9973, 'prob_fake': 0.0027}
{'prediction': 'FAKE', 'confidence': 0.9993, 'prob_real': 0.0007, 'prob_fake': 0.9993}


In [34]:
print("\nSTEP 10: Final evaluation with best checkpoint...")

# Load the state dictionary to CPU first
state_dict = torch.load(CONFIG['save_path'], map_location='cpu')
model.load_state_dict(state_dict)

_, final_f1, final_acc, final_preds, final_labels = eval_epoch(
    model, test_loader, criterion
)

print(f"\nFinal Test Macro-F1 : {final_f1:.4f}")
print(f"Final Test Accuracy  : {final_acc:.4f}")
print("\nClassification Report:")
print(classification_report(final_labels, final_preds, target_names=['Real', 'Fake']))
print("Confusion Matrix:")
print(confusion_matrix(final_labels, final_preds))

pd.DataFrame(history).to_csv('training_history.csv', index=False)
print(f"\nSaved -> training_history.csv")
print(f"Saved -> {CONFIG['save_path']}")


STEP 10: Final evaluation with best checkpoint...

Final Test Macro-F1 : 0.9632
Final Test Accuracy  : 0.9682

Classification Report:
              precision    recall  f1-score   support

        Real       0.97      0.98      0.98      8466
        Fake       0.96      0.94      0.95      3938

    accuracy                           0.97     12404
   macro avg       0.97      0.96      0.96     12404
weighted avg       0.97      0.97      0.97     12404

Confusion Matrix:
[[8308  158]
 [ 236 3702]]

Saved -> training_history.csv
Saved -> xlmroberta_urdu_best.pt
